# [LAB] MCP 서버와 클라이언트 실습

MCP 서버를 직접 만들어 터미널에 띄우고, 노트북의 Agent가 그 서버에 연결해 도구를 쓰게 합니다.

- **2장 MCP 서버 만들고 띄우기**: 서버 코드를 셀로 나눠 읽고, 터미널에서 서버를 띄웁니다. 서버 파일에서 도구 설명 한 줄을 바꿔 봅니다.
- **3장 클라이언트로 연결하기**: 서버에 연결해 도구를 가져오고, 그 도구로 Agent 그래프를 만듭니다. 보기를 골라 코드를 완성합니다. 마지막에 연결 방식을 SSE에서 stdio로 한 줄 바꿔 비교합니다.

셀은 위에서부터 차례로 실행합니다. 3장은 2장의 서버가 터미널에 켜져 있어야 동작합니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. MCP 서버 만들고 띄우기
  - 2.1 서버 만들기
  - 2.2 Tool 정의하기
  - 2.3 Resource 정의하기
  - 2.4 Prompt 정의하기
  - 2.5 실행 방식 정하기
- 3. 클라이언트로 연결하기
  - 3.1 연결 방식 고르기
  - 3.2 클라이언트 만들기
  - 3.3 도구 목록 가져오기
  - 3.4 LLM과 도구 노드에 도구 연결하기
  - 3.5 agent 노드
  - 3.6 그래프 연결과 컴파일
  - 3.7 질문하기
- 4. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/01-mcp/01_mcp_server.py`·`02_mcp_client.py`(둘 다 raw). 서버는 `%%writefile`로 노트북 폴더에 `mcp_server.py`를 쓰고 터미널에서 띄웁니다(사용자 결정). 생성된 `course/Chapter02/notebooks/mcp_server.py`는 강사 저장소에서 실행한 뒤 커밋에 섞이지 않게 지웁니다(.gitignore 등록 여부는 확인하지 않았습니다).
- 노트북에서만 원본을 손봤습니다(patch, 원본 파일은 그대로).
  - 서버: 머리 docstring의 파일 이름·Claude Code 등록 명령을 한 줄 설명으로 줄였습니다. 채울 줄이 없는 설명성 TODO 두 줄은 솔루션처럼 일반 주석으로 바꾸거나 뺐습니다. 이전 강사 메모 주석(솔루션본의 슬랙 MCP 예 등)은 넣지 않았습니다. 서버 바인딩은 원본·슬라이드 p8의 `0.0.0.0` 대신 `127.0.0.1`로 바꿨습니다(2026-09-29 리뷰). 클라이언트는 모두 localhost로 접속하므로 충분하고, `0.0.0.0`이면 Windows 방화벽 허용 창이 뜨고 같은 망의 다른 PC가 수강생 서버에 접속할 수 있습니다. 코드에 두 값의 차이를 한 줄 주석으로 적었습니다.
  - 클라이언트: `async def main()`과 `asyncio.run(main())`을 풀어 셀마다 top-level `await`로 실행합니다(노트북은 이미 이벤트 루프가 돌아 `asyncio.run`이 오류). raw는 stdio가 켜져 있고 SSE가 주석 처리된 알려진 결함이 있어, 두 설정을 `CONNECTIONS`에 모두 두고 `TRANSPORT` 한 줄로 고르게 했습니다. 기본은 슬라이드와 같은 SSE입니다. stdio의 `command`는 `sys.executable`(노트북 커널의 파이썬), 경로는 노트북 폴더 기준 `mcp_server.py`입니다. TODO 보조 주석, 쓰지 않는 `# local_tools = ...`, 노드 설명 메모, 연결 형식 설명 문자열은 뺐습니다.
  - 출력은 raw의 `A: …` 한 줄을 씁니다(솔루션은 `print(result)`로 상태 전체를 출력해 깁니다).
- 사외 실측(2026-09-28, `COURSE_ENV=out`, OpenAI): SSE 연결로 도구 5개, 질문 3개(시간·BMI·할 일) 약 10초. stdio는 약 20초. 사내 게이트웨이의 `bind_tools` 지원과 소요 시간은 미확인입니다.
- 목차의 4장(stdio로 바꿔 보기)은 코드 셀 없이 3장 셀을 다시 실행하는 장이라, 빌더 구조상 3장 안(3.1 CHANGE_HERE)으로 넣었습니다.

</details>

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. MCP 서버 만들고 띄우기

FastMCP로 MCP 서버를 만듭니다. 서버는 노트북 셀 안에서 돌리지 않고 터미널에서 띄웁니다. 서버는 켜 둔 동안 계속 요청을 기다리기 때문에, 노트북 셀에서 돌리면 다른 셀을 실행할 수 없습니다.

이 폴더에는 완성된 서버 파일 `mcp_server.py`가 미리 들어 있습니다. 아래 셀은 그 파일의 코드를 나눠 보여 주는 곳이라 차례로 읽고 실행해 봅니다. 셀을 실행해도 서버 파일은 바뀌지 않습니다.

서버에는 3대 Primitive가 모두 들어 있습니다.

- **Tool 5개:** 현재 시간, BMI 계산, 할 일 추가·목록·완료
- **Resource 1개:** 서버 정보(`config://app-info`)
- **Prompt 1개:** 할 일 요약 요청 템플릿

셀을 다 본 뒤, 아래 안내대로 터미널에서 서버를 띄웁니다.

### 2.1 서버 만들기

In [ ]:
"""
MCP 서버 만들기 (FastMCP)

터미널에서 실행한다. 기본은 HTTP/SSE 모드(포트 8000), --stdio를 붙이면 stdio 모드로 실행한다.
"""

import sys
from datetime import datetime

from fastmcp import FastMCP

# FastMCP: Anthropic MCP SDK 위에서 동작하는 고수준 래퍼.
# @mcp.tool(), @mcp.resource(), @mcp.prompt() 데코레이터로 기능을 정의한다.
mcp = FastMCP("AI 강의 튜토리얼 서버")

**성공 기준**

- 🟢 셀이 오류 없이 실행된다(서버 객체만 만들어 출력은 없다)

### 2.2 Tool 정의하기

**✍️ CHANGE_HERE: 도구 설명을 바꿔 봅니다**

먼저 그대로 두고 3장의 도구 목록까지 실행해 봅니다. 서버는 이 셀이 아니라 폴더의 `mcp_server.py` 파일로 돌기 때문에, 바꿀 때는 그 파일을 고칩니다. VS Code 왼쪽 탐색기에서 노트북과 같은 폴더의 `mcp_server.py`를 열고, `get_current_time` 함수 아래 따옴표 안 설명을 바꿔(예: "지금 한국 날짜와 시각을 알려 준다.") 저장합니다(`Ctrl + S`). 터미널에서 **Ctrl + C**로 서버를 끄고 같은 명령으로 다시 띄운 뒤, 3장의 연결 셀과 도구 목록 셀을 다시 실행합니다. 서버의 도구 설명은 클라이언트가 가져가 LLM이 도구를 고르는 근거로 씁니다.

In [ ]:
# ── Tool: LLM이 호출할 수 있는 함수 ──────────────────────────────────────────
@mcp.tool()
def get_current_time() -> str:
    # ✍️ CHANGE_HERE: 도구 설명을 바꿔 봅니다
    """현재 날짜와 시간을 반환한다."""
    return datetime.now().strftime("%Y년 %m월 %d일 %H:%M:%S")


@mcp.tool()
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """체중(kg)과 키(cm)로 BMI를 계산한다."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m**2)
    if bmi < 18.5:
        category = "저체중"
    elif bmi < 23:
        category = "정상"
    elif bmi < 25:
        category = "과체중"
    else:
        category = "비만"
    return {"bmi": round(bmi, 1), "category": category}


_tasks: list[dict] = []


@mcp.tool()
def add_task(title: str, priority: int = 1) -> str:
    """할 일 목록에 새 항목을 추가한다. priority는 1(낮음)~3(높음)."""
    task = {"id": len(_tasks) + 1, "title": title, "priority": priority, "done": False}
    _tasks.append(task)
    return f"할 일 추가됨: [{task['id']}] {title} (우선순위: {priority})"


@mcp.tool()
def list_tasks() -> list[dict]:
    """현재 할 일 목록을 반환한다."""
    return _tasks if _tasks else [{"message": "할 일이 없습니다."}]


@mcp.tool()
def complete_task(task_id: int) -> str:
    """할 일을 완료 처리한다."""
    for task in _tasks:
        if task["id"] == task_id:
            task["done"] = True
            return f"완료: {task['title']}"
    return f"ID {task_id}를 찾을 수 없습니다."

**성공 기준**

- 🟢 셀이 오류 없이 실행된다
- 🟢 (파일의 설명을 바꾼 뒤) 서버를 다시 띄우고 3장 도구 목록을 다시 출력하면 `get_current_time` 옆 설명이 바꾼 글로 보인다
- ⛔ 설명이 그대로다 → `mcp_server.py` 파일을 저장했는지, 서버를 Ctrl + C로 끄고 다시 띄웠는지, 3장 도구 목록 셀을 다시 실행했는지 봅니다

### 2.3 Resource 정의하기

In [ ]:
# ── Resource: LLM에게 제공할 데이터/문서 ──────────────────────────────────────
@mcp.resource("config://app-info")
def get_app_info() -> str:
    """이 MCP 서버의 기본 정보를 제공한다."""
    return """
    서버명: AI 강의 튜토리얼 MCP 서버
    버전: 1.0.0
    기능: 시간 조회, BMI 계산, 할 일 관리
    """

### 2.4 Prompt 정의하기

In [ ]:
# ── Prompt: 자주 쓰는 프롬프트 템플릿 ────────────────────────────────────────
@mcp.prompt()
def task_summary_prompt(user_name: str) -> str:
    """할 일 목록 요약 요청 프롬프트를 생성한다."""
    return f"{user_name}님의 할 일 목록을 우선순위별로 정리하고 오늘 집중해야 할 항목을 추천해줘."

### 2.5 실행 방식 정하기

아래는 서버를 터미널에서 띄울 때만 실행되는 부분입니다. 노트북에서는 실행하지 않고 읽어만 봅니다.

```python
if __name__ == "__main__":
    if "--stdio" in sys.argv:
        # stdio 모드: Claude Code, Cursor 등 MCP Host와 직접 연결
        mcp.run(transport="stdio")
    else:
        print("MCP 서버 시작: http://localhost:8000/sse")
        print("종료하려면 Ctrl+C")
        # HTTP/SSE 모드: 클라이언트가 localhost:8000으로 연결
        # 127.0.0.1: 이 PC에서 오는 요청만 받는다. 0.0.0.0이면 다른 PC의 요청도 받는다.
        mcp.run(transport="sse", host="127.0.0.1", port=8000)
```

이 폴더에는 정답이 모두 들어간 서버 파일(`mcp_server.py`)이 미리 들어 있습니다. 2.1부터 위 셀들은 그 파일의 코드를 나눠 보여 주는 곳입니다. 셀을 실행해 오류가 없는지 확인합니다(보기가 있는 셀은 먼저 고릅니다). 셀이 파일을 고치지는 않습니다.

서버는 터미널에서 이 완성 파일로 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/mcp_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다. 서버를 끌 때는 터미널에서 **Ctrl + C**를 누릅니다.

**성공 기준** (터미널 화면)

- 🟢 `MCP 서버 시작: http://localhost:8000/sse`와 `종료하려면 Ctrl+C` 두 줄이 보인다
- 🟢 그 아래 FastMCP 안내 상자(서버 이름 `AI 강의 튜토리얼 서버`가 적힌 상자)가 나오고, 맨 끝 줄에 `Uvicorn running on`과 `(Press CTRL+C to quit)`가 보인다
- 🟢 명령이 끝나지 않고 기다린다(새 명령을 입력하는 줄이 돌아오지 않는다)
- ⛔ 명령이 곧바로 끝나고 오류가 나온다 → 오류 메시지 전체를 AI에게 보여 줍니다
- ⛔ 오류에 `address already in use` 또는 `10048`이 보인다 → 이미 켜 둔 서버가 있습니다. 앞서 서버를 띄운 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

**볼 지점**
- 서버를 띄운 터미널은 3장이 끝날 때까지 켜 둡니다. 3장에서 도구를 호출할 때마다 이 터미널에 요청 기록(`POST /messages/...` 같은 줄)이 늘어납니다.
- 안내 상자에 새 버전 안내(`Update available`)가 함께 나올 수 있습니다. 실습과 관계없으니 그대로 둡니다.

<details><summary>강사용 메모</summary>

- 사외 실측(macOS, fastmcp 3.4.7): 시작 문구 두 줄 → FastMCP 배너(로고, `FastMCP 3.4.7`, `Server: AI 강의 튜토리얼 서버, 3.4.7`, `Deploy free` 안내) → `Update available: 4.0.10` 상자 → `INFO Starting MCP server 'AI 강의 튜토리얼 서버' with transport 'sse' on http://127.0.0.1:8000/sse` → uvicorn 시작 4줄(2026-09-29 바인딩을 `127.0.0.1`로 바꾼 뒤 기준). 강의장 PC(Windows 콘솔)에서 배너 글자가 깨지는지는 미확인입니다. `127.0.0.1` 바인딩이라 방화벽 허용 창은 뜨지 않을 것으로 보지만 강의장 PC에서 확인합니다.
- 터미널이 저장소 맨 위 폴더에서 열린다는 전제(VS Code 기본)입니다. `uv run`이 `.venv`를 찾지 못하면 저장소 맨 위로 `cd`한 뒤 다시 실행합니다.
- 원본 서버는 `.env`를 읽지 않지만, 터미널에 `python-dotenv could not parse statement` 경고가 나올 수 있습니다(fastmcp가 `.env`를 읽음). 강사 맥 `.env`의 깨진 줄 때문이며 수강생 PC와 무관할 수 있습니다.

</details>

## 3. 클라이언트로 연결하기

노트북에서 Agent(Host)를 만들고, 그 안의 Client로 2장의 서버에 연결합니다. 서버의 도구를 가져와 LLM에 넘기고, 과정 1에서 만든 것과 같은 모양의 LangGraph 그래프(agent ⇄ tools)로 질문에 답하게 합니다.

- **연결:** 클라이언트 만들기, 도구 목록 가져오기
- **Agent 그래프:** LLM과 도구 노드에 도구 연결하기, agent 노드, 조건부 엣지, 컴파일

서버와 주고받는 코드 앞에는 `await`가 붙습니다. 응답이 올 때까지 기다린다는 표시입니다. 노트북 셀에서는 `await`를 바로 쓸 수 있습니다.

마지막 질문 셀은 앞의 셀에서 **모두 보기를 골라 실행한 뒤에** 동작합니다.

**왜 배우나:** 도구를 서버에서 가져온다는 점만 다르고, 나머지는 과정 1의 `bind_tools`·ToolNode·조건부 엣지를 그대로 씁니다. 도구가 어디에 있든 Agent 쪽 코드는 같다는 것을 직접 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고 설정 파일을 읽습니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
MCP 클라이언트 실습: 터미널에 켜 둔 MCP 서버에 연결해 LangGraph Agent가 도구로 사용한다.
"""

import sys
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv();

### 3.1 연결 방식 고르기

**✍️ CHANGE_HERE: 연결 방식을 "sse"에서 "stdio"로 바꿔 봅니다**

처음에는 그대로(`"sse"`) 두고 3장 끝까지 실행합니다. 모두 동작하면 이 줄을 `"stdio"`로 바꾸고 이 셀부터 질문하기 셀까지 차례로 다시 실행합니다. SSE는 터미널에 켜 둔 서버에 주소로 접속하고, stdio는 노트북이 서버 파일을 직접 실행해 주고받습니다. Windows에서는 노트북이 서버 파일을 직접 실행하지 못해 도구 목록 가져오기 셀에서 오류가 납니다. 이 오류는 정상입니다. 비교가 끝나면(Windows는 오류를 확인하면) `"sse"`로 되돌립니다.

In [ ]:
# ── MCP 서버 연결 설정 ────────────────────────────────────────────────────────
# SSE(Server-Sent Events): HTTP 기반 MCP 연결 방식.
# 서버를 HTTP 모드로 실행하면 /sse 엔드포인트가 열린다. 터미널에 켜 둔 서버에 주소로 접속한다.
# stdio: 노트북이 서버 파일을 직접 실행해 표준 입출력으로 주고받는다.
CONNECTIONS = {
    "sse": {"transport": "sse", "url": "http://localhost:8000/sse"},
    "stdio": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["mcp_server.py", "--stdio"],
    },
}
# ✍️ CHANGE_HERE: 연결 방식을 "sse"에서 "stdio"로 바꿔 봅니다
TRANSPORT = "sse"
MCP_CONFIG = {"tutorial": CONNECTIONS[TRANSPORT]}

**성공 기준**

- 🟢 셀이 오류 없이 끝난다(설정만 만들어 출력은 없다)
- 🟢 (Windows에서 "stdio"로 바꾼 뒤) 도구 목록 가져오기 셀이 `NotImplementedError`와 `UnsupportedOperation: fileno` 오류로 멈추면 정상입니다. `"sse"`로 되돌리고 넘어갑니다
- 🟢 (macOS·Linux에서 "stdio"로 바꾸고 다시 실행한 뒤) 서버 터미널을 Ctrl+C로 꺼 두어도 도구 목록에 같은 도구 5개가 나온다
- 🟢 (macOS·Linux에서 "stdio"로 바꾸고 다시 실행한 뒤) BMI 답은 SSE 때와 같지만, 할 일 질문에는 추가했는데 목록이 비어 있다는 식으로 답한다
- ⛔ (macOS·Linux에서 "stdio"로 바꾼 뒤) 오류가 난다 → 이 PC에서 stdio 방식이 서버 프로그램을 띄우지 못한 것입니다. 오류를 AI에게 보여 원인을 물어보고, 풀리지 않으면 `"sse"`로 되돌려 넘어갑니다

### 3.2 클라이언트 만들기

**✅ MCP 설정으로 MultiServerMCPClient를 생성해 주세요.**

- 🟢 3.3 도구 목록 가져오기 셀까지 오류 없이 도구 5줄이 나온다
- ⛔ 3.2에서 `TypeError`가 나거나 3.3에서 `AttributeError`·`ValueError: Configuration error`가 보인다 → 다른 보기를 고릅니다

In [ ]:
# ✅ CHOOSE_HERE: MCP 설정으로 MultiServerMCPClient를 생성해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# client = MultiServerMCPClient(TRANSPORT)               # 1️⃣ 연결 방식 글자를 넘김
# client = await MultiServerMCPClient(MCP_CONFIG)        # 2️⃣ 클라이언트 만들기를 기다림
# client = MultiServerMCPClient(MCP_CONFIG)              # 3️⃣ 서버 이름이 붙은 설정을 넘김
# client = MultiServerMCPClient(MCP_CONFIG["tutorial"])  # 4️⃣ 서버 하나의 연결 설정만 넘김

<details><summary>정답 보기</summary>

위 셀에서 만든 연결 설정 `MCP_CONFIG`로 클라이언트를 만들어 변수 `client`에 담는 한 줄입니다. 이 줄은 서버에 바로 접속하지 않습니다. 설정만 들고 있다가 다음 셀에서 도구를 가져올 때 접속합니다.

**정답: 3️⃣**

- 1️⃣ `TRANSPORT`는 `"sse"`라는 글자일 뿐입니다. 3.2는 통과하지만 3.3에서 도구를 가져올 때 `AttributeError: 'str' object has no attribute 'items'`가 납니다.
- 2️⃣ 클라이언트를 만드는 일은 서버와 주고받지 않아 기다릴 것이 없습니다. 3.2에서 `TypeError: object MultiServerMCPClient can't be used in 'await' expression`이 납니다.
- 3️⃣ `MCP_CONFIG`는 서버 이름(`tutorial`)과 그 연결 설정을 묶은 것입니다. 클라이언트는 이 설정을 들고 있다가 3.3에서 도구를 가져올 때 서버에 접속합니다.
- 4️⃣ 서버 이름 없이 연결 설정만 넘겨 `transport`·`url`을 서버 이름으로 읽습니다. 3.3에서 `ValueError: Configuration error: Missing 'transport' key in server configuration`이 납니다.

</details>

### 3.3 도구 목록 가져오기

**✅ MCP 서버에서 도구 목록을 가져와 주세요.**

- 🟢 `=== 사용 가능한 MCP 도구 ===` 아래에 `- get_current_time: …`으로 시작하는 도구 5줄이 나온다
- ⛔ `TypeError`나 `AttributeError`가 보인다 → 다른 보기를 고릅니다(`ConnectError`면 2장의 서버 터미널을 켭니다)

In [ ]:
# get_tools()는 서버에 연결해 도구 목록을 가져온다.
# ✅ CHOOSE_HERE: MCP 서버에서 도구 목록을 가져와 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# tools = await client.get_tools()                # 1️⃣ 도구 목록을 받아 올 때까지 기다림
# tools = client.get_tools()                      # 2️⃣ await 없이 부름
# tools = await client.get_tools                  # 3️⃣ 괄호 없이 기다림
# tools = await client.get_resources("tutorial")  # 4️⃣ 서버의 Resource를 가져옴
print("=== 사용 가능한 MCP 도구 ===")
for t in tools:
    print(f"  - {t.name}: {t.description}")
print()

<details><summary>정답 보기</summary>

서버에 접속해 도구 이름과 설명을 읽어 LangChain Tool 목록으로 받아 `tools`에 담는 한 줄입니다. 서버와 주고받는 동안 기다려야 하므로 `await`가 필요합니다. 아래 줄들이 받은 도구를 하나씩 출력합니다.

**정답: 1️⃣**

- 1️⃣ 서버에 접속해 도구 5개를 받아 옵니다. 서버와 주고받는 일이라 `await`로 응답을 기다립니다.
- 2️⃣ 응답을 기다리지 않아 도구 목록 대신 아직 실행되지 않은 작업(coroutine)이 담깁니다. 제목 줄만 나오고 `TypeError: 'coroutine' object is not iterable`이 납니다.
- 3️⃣ 괄호가 없어 함수를 부르지 않고 함수 자체를 기다립니다. `TypeError: object method can't be used in 'await' expression`이 납니다.
- 4️⃣ 도구가 아니라 Resource(`config://app-info`)를 가져옵니다. Resource는 데이터라서 도구 이름이 없습니다. 제목 줄만 나오고 `AttributeError: 'Blob' object has no attribute 'name'`이 납니다.

</details>

### 3.4 LLM과 도구 노드에 도구 연결하기

**✅ 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.**

- 🟢 3.7에서 시간·BMI 질문의 답에 실제 시각과 BMI 22.9가 들어 있다
- ⛔ 3.7에서 시간을 "확인할 수 없다"고 답하거나 `AttributeError`가 보인다 → 다른 보기를 고릅니다

In [ ]:
# ✅ CHOOSE_HERE: 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# llm = get_llm()                    # 1️⃣ 도구 노드만 만듦
# tool_node = ToolNode(tools)        # 1️⃣
#
# llm = get_llm().bind_tools(tools)  # 2️⃣ ToolNode를 도구 없이 그대로 씀
# tool_node = ToolNode               # 2️⃣
#
# llm = get_llm().bind_tools(tools)  # 3️⃣ 빈 목록으로 도구 노드를 만듦
# tool_node = ToolNode([])           # 3️⃣
#
# llm = get_llm().bind_tools(tools)  # 4️⃣ 모델과 도구 노드에 같은 도구를 줌
# tool_node = ToolNode(tools)        # 4️⃣

<details><summary>정답 보기</summary>

두 줄입니다. 하나는 `get_llm()`으로 만든 모델에 서버의 도구 목록을 알려 주어 `llm`에 담고, 다른 하나는 같은 도구 목록으로 도구를 실제로 실행하는 노드 `tool_node`를 만듭니다. 과정 1의 Tool Calling과 LangGraph에서 쓴 방법과 같습니다.

**정답: 4️⃣**

- 1️⃣ 모델이 도구가 있는지 모릅니다. 오류 없이 3.7에서 시간은 "확인할 수 없다", BMI는 계산식만 답합니다. 할 일은 추가하지 않았는데 추가했다고 지어내 답하기도 합니다.
- 2️⃣ `ToolNode`에 도구 목록을 넘기지 않아 노드가 만들어지지 않습니다. 3.7 첫 질문에서 `AttributeError: 'function' object has no attribute 'name'`이 납니다.
- 3️⃣ 모델은 도구를 부르지만 도구 노드에 실행할 도구가 없습니다. 오류 없이 3.7에서 시간은 "확인할 수 없다", 할 일은 "도구 오류가 발생했다"고 답합니다.
- 4️⃣ 모델은 어떤 도구를 부를 수 있는지 알고, 도구 노드는 그 도구를 실제로 실행합니다. 3.7에서 시각, BMI 22.9, 할 일 목록이 나옵니다.

</details>

### 3.5 agent 노드

**✅ State의 messages를 도구가 연결된 LLM에 전달해 주세요.**

- 🟢 3.7에서 질문마다 마지막에 답이 든 `Ai Message` 칸이 나온다
- ⛔ 3.7에서 `Ai Message` 칸 없이 `Human Message`만 나오거나 `ValueError`가 보인다 → 다른 보기를 고릅니다

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    # ✅ CHOOSE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # response = llm.invoke(state)              # 1️⃣ State 전체를 LLM에 넘김
    # return {"messages": [response]}           # 1️⃣
    #
    # response = llm.invoke(state["messages"])  # 2️⃣ 메시지 목록을 넘기고 응답을 담음
    # return {"messages": [response]}           # 2️⃣
    #
    # response = llm.invoke(state["messages"])  # 3️⃣ 받은 State를 그대로 돌려줌
    # return state                              # 3️⃣
    #
    # response = llm.invoke(state["messages"])  # 4️⃣ 응답의 글자만 담음
    # return {"messages": [response.content]}   # 4️⃣
    pass  # 그대로 둡니다

<details><summary>정답 보기</summary>

`agent_node`는 Agent가 한 번 생각하는 단계입니다. 두 줄입니다. 상태에 쌓인 메시지를 도구가 연결된 `llm`에 보내 응답을 받고, 그 응답을 `messages`에 더하도록 딕셔너리로 돌려줍니다.

**정답: 2️⃣**

- 1️⃣ State 딕셔너리를 통째로 넘깁니다. LLM은 메시지 목록을 받아야 해서 3.7에서 `ValueError: Invalid input type <class 'dict'>`가 납니다.
- 2️⃣ 메시지 목록을 넘겨 응답을 받고, 대괄호로 감싸 `messages`에 담습니다. 응답에 도구 호출 요청이 들어 있으면 조건부 엣지가 tools로 보냅니다.
- 3️⃣ 응답을 버리고 받은 State를 그대로 돌려줍니다. 오류 없이 3.7에 `Ai Message` 칸 없이 질문(`Human Message`)만 나옵니다.
- 4️⃣ 응답의 글자만 담아 도구 호출 요청이 사라집니다. 글자만 담으면 `Human Message`로 쌓이고, 도구를 부를 때는 그 글자가 비어 있어서 오류 없이 3.7에 빈 `Human Message` 칸이 나오고 끝납니다.

</details>

### 3.6 그래프 연결과 컴파일

**✅ tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

- 🟢 3.7에서 할 일 질문의 마지막 `Ai Message`에 "LangGraph 공부"가 든 목록이 나온다
- ⛔ 3.7이 `Tool Message` 칸이나 `Tool Calls:`가 든 `Ai Message` 칸에서 끝나거나 3.6에서 `TypeError`가 보인다 → 다른 보기를 고릅니다

**✅ 그래프를 컴파일해 주세요.**

- 🟢 3.7이 오류 없이 세 질문에 모두 답한다
- ⛔ 3.6이나 3.7에서 `ValueError`·`TypeError`·`AttributeError`가 보인다 → 다른 보기를 고릅니다

In [ ]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent");
# ✅ CHOOSE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# builder.add_conditional_edges("agent", tools_condition)    # 1️⃣ agent 뒤 분기 + tools→agent
# builder.add_edge("tools", "agent")                         # 1️⃣
#
# builder.add_conditional_edges("agent", tools_condition)    # 2️⃣ agent 뒤 분기만 둠
#
# builder.add_conditional_edges("tools", tools_condition)    # 3️⃣ tools 뒤 분기 + tools→agent
# builder.add_edge("tools", "agent")                         # 3️⃣
#
# builder.add_conditional_edges("agent", tools_condition())  # 4️⃣ tools_condition을 불러 그 결과로 분기
# builder.add_edge("tools", "agent")                         # 4️⃣

# ✅ CHOOSE_HERE: 그래프를 컴파일해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# app = builder.compile              # 1️⃣ 괄호 없이 compile을 담음
# app = await builder.compile()      # 2️⃣ 컴파일을 기다림
# app = builder.compile()            # 3️⃣ 조립한 builder를 컴파일
# app = StateGraph(State).compile()  # 4️⃣ 새 그래프를 컴파일

<details><summary>정답 보기</summary>

**tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

두 줄입니다. agent 노드 다음에 LLM이 도구 호출을 요청했으면 tools로, 아니면 끝(END)으로 가게 조건부 엣지를 답니다. 그리고 tools 노드가 도구를 실행한 뒤에는 다시 agent로 돌아와 결과를 보고 답하게 합니다.

**정답: 1️⃣**

- 1️⃣ agent 뒤에서 도구 호출이 있으면 tools로, 없으면 끝으로 갑니다. 도구를 실행한 뒤 agent로 돌아와 결과를 보고 답합니다.
- 2️⃣ 도구를 실행한 뒤 돌아올 길이 없어 그래프가 끝납니다. 오류 없이 3.7이 `[{'type': 'text', 'text': '{"bmi":22.9,...}'}]` 같은 도구 결과가 든 `Tool Message` 칸에서 끝납니다.
- 3️⃣ 분기를 agent가 아니라 tools 뒤에 달아 agent 다음에 갈 곳이 없습니다. 도구를 부르기 전에 끝나서, 오류 없이 3.7이 도구 호출(`Tool Calls:`)이 든 `Ai Message` 칸에서 끝납니다.
- 4️⃣ `tools_condition`에는 함수 자체를 넘깁니다. 괄호를 붙이면 그 자리에서 불러 3.6에서 `TypeError: tools_condition() missing 1 required positional argument: 'state'`가 납니다.

**그래프를 컴파일해 주세요.**

노드와 엣지를 다 붙인 `builder`를 실행할 수 있는 그래프로 만들어 `app`에 담는 한 줄입니다. 다음 셀이 `app.ainvoke(...)`로 이 그래프를 실행합니다.

**정답: 3️⃣**

- 1️⃣ 괄호가 없어 컴파일하지 않고 `compile` 함수 자체를 담습니다. 3.7에서 `AttributeError: 'function' object has no attribute 'ainvoke'`가 납니다.
- 2️⃣ 컴파일은 서버와 주고받지 않아 기다릴 것이 없습니다. 3.6에서 `TypeError: object CompiledStateGraph can't be used in 'await' expression`이 납니다.
- 3️⃣ 노드와 엣지를 붙인 설계도(`builder`)를 실행할 수 있는 `app`으로 만듭니다. 3.7에서 `app.ainvoke(...)`로 실행합니다.
- 4️⃣ 노드가 하나도 없는 새 그래프를 컴파일합니다. 3.6에서 `ValueError: Graph must have an entrypoint`가 납니다.

</details>

### 3.7 질문하기

**✍️ CHANGE_HERE: 첫 질문을 도구가 필요 없는 질문으로 바꿔 봅니다**

보기를 모두 골라 세 질문의 답을 본 뒤에 합니다. `# ✍️ CHANGE_HERE:` 아래 따옴표 안 질문을 도구가 필요 없는 질문(예: "MCP가 뭐야?")으로 바꿔 이 셀을 다시 실행합니다. LLM이 도구를 호출하지 않기로 판단하면 조건부 엣지가 tools를 거치지 않고 바로 끝냅니다.

In [ ]:
# 도구를 사용해 요청 처리
questions = [
    # ✍️ CHANGE_HERE: 첫 질문을 도구가 필요 없는 질문으로 바꿔 봅니다
    "지금 몇 시야?",
    "체중 70kg, 키 175cm면 BMI가 얼마야?",
    '할 일 목록에 "LangGraph 공부" (우선순위 3)를 추가하고 현재 목록을 보여줘.',
]

for question in questions:
    mcp_graph_result = await app.ainvoke(
        {
            "messages": [HumanMessage(content=question)],
        }
    )
    results = mcp_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print()

**성공 기준**

- 🟢 (바꾼 뒤) 도구가 필요 없는 질문에는 시각 같은 도구 결과 없이 LLM이 바로 답한다
- ⛔ (바꾼 뒤) 셀이 끝나지 않고 계속 돈다 → 셀 실행을 중지(■)하고 3.6에서 고른 조건부 엣지 보기를 다시 봅니다

**볼 지점** (질문하기 셀)
- 질문마다 `Human Message` → `Ai Message`(도구 호출) → `Tool Message`(서버 도구의 결과) → `Ai Message`(답) 칸이 나옵니다. 세 질문 모두 LLM이 서버의 도구를 호출해 얻은 값으로 답합니다.
  - 시간 질문: 지금 날짜와 시각
  - BMI 질문: BMI 22.9, 정상 범위
  - 할 일 질문: "LangGraph 공부"(우선순위 3)가 든 목록
- 서버 터미널에 도구 호출마다 요청 기록이 늘어납니다. Agent는 노트북에서, 도구는 터미널의 서버에서 실행된다는 뜻입니다.
- 서버를 켠 채로 질문하기 셀을 다시 실행하면 할 일 목록에 "LangGraph 공부"가 하나씩 더 쌓입니다. 서버가 켜져 있는 동안 목록을 기억하기 때문이며 정상입니다. 서버를 껐다 켜면 비워집니다.
- 답의 말투와 모양은 실행할 때마다 조금씩 다릅니다.

**stdio로 바꿔 보기**

3.1 연결 방식 고르기 셀의 `TRANSPORT`를 `"stdio"`로 바꾸고 질문하기 셀까지 다시 실행해 결과를 SSE 때와 비교합니다(성공 기준은 3.1 셀 아래에 있습니다). Windows에서는 도구 목록 가져오기 셀에서 오류가 나며 멈추는 것이 정상입니다. 그때는 `"sse"`로 되돌리고 아래 설명으로 차이를 봅니다. stdio에서는 노트북이 도구를 호출할 때마다 서버 프로그램을 새로 실행하고 끝냅니다. 그래서 할 일을 추가한 서버와 목록을 보여 주는 서버가 서로 다른 서버가 되어, 추가한 할 일이 목록에 보이지 않습니다.

<details><summary>강사용 메모</summary>

- 사외 실측(2026-09-28): SSE 약 10초. Q1 "지금은 2026년 9월 28일 밤 9시 56분이에요.", Q2 "BMI는 22.9 … 정상 범위", Q3 `add_task`·`list_tasks` 호출 후 목록 출력.
- stdio 실측: 약 20초. 도구 목록 5개 같음. 스크립트로 실행하면 FastMCP 배너가 5번(도구 가져오기 1 + 도구 호출 4) 나오지만, nbconvert 실행에서는 셀 출력에 나오지 않았습니다(서버의 stderr가 커널 쪽으로 감). VS Code 셀 출력에 배너가 보이는지는 미확인이라 성공 기준에 넣지 않았습니다. Q3 답은 두 번 모두 "추가했습니다. 현재 목록에는 할 일이 없습니다"는 식(스크립트 실행 1회, 노트북 실행 1회). langchain-mcp-adapters 0.3.2는 세션 없이 가져온 도구를 호출할 때마다 새 세션(stdio면 새 서버 프로세스)을 엽니다. SSE는 서버 하나가 계속 떠 있어 목록이 남습니다. 조용히 틀리는 것처럼 보이지 않게 성공 기준과 볼 지점에 미리 적었습니다.
- Windows 노트북 커널에서는 stdio가 반드시 실패합니다(2026-09-29 설치 패키지 코드로 확인, Windows 실행은 미확인). ipykernel 7.3.0이 Windows에서 Selector 이벤트 루프를 써서 asyncio 하위 프로세스가 `NotImplementedError`를 내고, mcp 1.30.0이 `subprocess.Popen` 대체 경로로 넘어가지만 `stderr=sys.stderr`(ipykernel OutStream, Windows에서는 fd가 없음)에서 `io.UnsupportedOperation: fileno`로 끝납니다. 그래서 3.1 성공 기준에 Windows에서는 이 오류가 정상이고 `"sse"`로 되돌린다고 적었습니다. stdio 비교는 강사 화면(macOS) 시연으로 보여 주는 것을 권합니다.
- LLM이 할 일 질문에서 `list_tasks`를 호출하지 않고 답할 수도 있습니다(실측 두 번은 모두 호출).
- 원본은 마지막 답만 `print(...content)`로 출력합니다. 노트북은 결과 변수를 그래프가 드러나는 이름(`mcp_graph_result`)으로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).

</details>

## 4. 정리

**이 노트북에서 한 것**
- FastMCP로 Tool·Resource·Prompt가 든 MCP 서버 코드를 셀로 나눠 읽고, 터미널에서 HTTP/SSE 모드로 띄웠습니다.
- `MultiServerMCPClient`로 서버에 연결하고 `get_tools`로 도구 5개를 가져왔습니다. Resource와 Prompt는 도구 목록에 들어오지 않습니다.
- 가져온 도구를 `bind_tools`와 ToolNode에 넘겨, 과정 1과 같은 agent ⇄ tools 그래프로 질문에 답하게 했습니다.
- 연결 방식을 stdio로 바꾸면 노트북이 서버를 직접 실행합니다. 호출할 때마다 서버가 새로 떠서 할 일 목록이 이어지지 않습니다. Windows 노트북에서는 stdio가 서버를 띄우지 못해 오류가 납니다.

실습이 끝나면 서버를 띄운 터미널에서 **Ctrl+C**로 서버를 끕니다.